# Data Cleaning Project

**Name:** _[Your Name Here]_
**Course / Assignment:** Data Cleaning — Professional Data Cleaning Skills Demonstration
**Date:** _[Submission Date]_
**Tech Stack:** Python, pandas, numpy, Jupyter Notebook

---

### Assignment Checklist Coverage
| # | Requirement | Section |
|---|---|---|
| 1 | Data quality report (nulls, duplicates, dtype issues, range anomalies) | Section 1 |
| 2 | Missing data handling, strategy justified per column | Section 6 |
| 3 | Duplicate removal, documented count | Section 3 |
| 4 | Standardisation (categorical values, date formats) | Section 4 |
| 5 | Outlier detection (IQR method), cap/remove/retain decisions documented | Section 5 |
| 6 | Data type correction | Sections 2 and 7 |
| 7 | Before vs. after summary table | Section 8 |
| 8 | Cleaned dataset saved to new CSV | Section 9 |

---


# Data Cleaning Project: Customer Purchase Dataset

**Objective:** Take a deliberately messy customer dataset and systematically transform it
into a clean, analysis-ready dataset, documenting every decision along the way.

**Dataset:** `messy_customer_data.csv` — a synthetic customer/purchase dataset (1,043 rows,
8 columns) built to reproduce the categories of mess found in real-world data: missing
values, exact and near-duplicate rows, inconsistent categorical formatting, mixed date
formats, numbers stored as text (currency symbols, stray whitespace), out-of-range values,
and inconsistent dtypes.

| Column | Description |
|---|---|
| `CustomerID` | Unique customer identifier (should be a string, e.g. `CUST00001`) |
| `Name` | Customer full name |
| `Age` | Customer age in years |
| `Gender` | Customer gender |
| `City` | Customer city |
| `SignupDate` | Date the customer signed up |
| `PurchaseAmount` | Amount spent, in USD |
| `SatisfactionScore` | Self-reported satisfaction, 1–5 scale |

**Tech stack:** Python, pandas, numpy, Jupyter Notebook


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = 'messy_customer_data.csv'
OUTPUT_PATH = 'cleaned_customer_data.csv'


## 1. Load Dataset & Data Quality Report

We load the raw CSV **without any parsing assumptions** (dates and numbers are left as
strings where the source data was inconsistent) so we can inspect the mess before touching
anything.

In [2]:
df_raw = pd.read_csv(RAW_PATH, dtype=str)  # load everything as string first, no silent coercion
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(10)


Shape: 1043 rows x 8 columns


,CustomerID,Name,Age,Gender,City,SignupDate,PurchaseAmount,SatisfactionScore
0,CUST00854,Mary Lopez,47.0,Male,Phoenix,24/06/2023,195.12,4.0
1,CUST00787,Robert Jones,33.0,MALE,Philadelphia,"January 18, 2023",289.34,4.0
2,CUST00315,Michael Gonzalez,41.0,male,Phoenix,01-07-2020,684.51,4.0
3,CUST00815,John Rodriguez,47.0,Male,Houston,03-25-2023,240.09,3.0
4,CUST00977,Joseph Thomas,49.0,female,San Diego,2024-04-08,42.15,3.0
5,CUST00010,Barbara Wilson,25.0,male,New York,"January 22, 2018",206.32,4.0
6,CUST00135,David Lopez,33,Male,Houston,"November 11, 2018",NaN,4.0
7,CUST00777,Michael Rodriguez,38.0,NaN,San Jose,2022-12-25,257.36,3.0
8,CUST00410,Mary Lopez,40.0,MALE,Phoenix,17-Aug-2020,244.16,1.0
9,CUST00413,Robert Hernandez,55.0,MALE,San Diego,08-24-2020,108.52,2.0


In [3]:
# --- Snapshot metrics we'll reuse in the Before/After summary at the end ---
before_stats = {
    'row_count': len(df_raw),
    'duplicate_rows': df_raw.duplicated().sum(),
    'null_counts': df_raw.isna().sum(),
    'dtypes': df_raw.dtypes.copy(),
}
before_stats


{'row_count': 1043,
 'duplicate_rows': np.int64(32),
 'null_counts': CustomerID            0
 Name                  5
 Age                  65
 Gender               32
 City                 26
 SignupDate           17
 PurchaseAmount       43
 SatisfactionScore    50
 dtype: int64,
 'dtypes': CustomerID           str
 Name                 str
 Age                  str
 Gender               str
 City                 str
 SignupDate           str
 PurchaseAmount       str
 SatisfactionScore    str
 dtype: object}

In [4]:
def data_quality_report(df, name='dataset'):
    print(f"{'='*60}\nDATA QUALITY REPORT: {name}\n{'='*60}")
    print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")

    print("\n--- Null counts per column ---")
    null_report = pd.DataFrame({
        'nulls': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2)
    })
    print(null_report)

    print(f"\n--- Duplicate rows ---")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")

    print(f"\n--- Data types ---")
    print(df.dtypes)

    print(f"\n--- Value range / anomaly scan (numeric-looking columns) ---")
    for col in df.columns:
        sample_non_null = df[col].dropna().astype(str)
        numeric_coerced = pd.to_numeric(sample_non_null, errors='coerce')
        pct_numeric = numeric_coerced.notna().mean() if len(sample_non_null) else 0
        if pct_numeric > 0.5:  # looks like it should be numeric
            print(f"  {col}: min={numeric_coerced.min()}, max={numeric_coerced.max()}, "
                  f"non-numeric entries={numeric_coerced.isna().sum() - df[col].isna().sum()}")
    return null_report

_ = data_quality_report(df_raw, 'RAW dataset')


DATA QUALITY REPORT: RAW dataset

Shape: 1043 rows, 8 columns

--- Null counts per column ---
                   nulls  null_pct
CustomerID             0      0.00
Name                   5      0.48
Age                   65      6.23
Gender                32      3.07
City                  26      2.49
SignupDate            17      1.63
PurchaseAmount        43      4.12
SatisfactionScore     50      4.79

--- Duplicate rows ---
Exact duplicate rows: 32

--- Data types ---
CustomerID           str
Name                 str
Age                  str
Gender               str
City                 str
SignupDate           str
PurchaseAmount       str
SatisfactionScore    str
dtype: object

--- Value range / anomaly scan (numeric-looking columns) ---
  Age: min=-20.0, max=999.0, non-numeric entries=-65
  PurchaseAmount: min=-250.0, max=99999.99, non-numeric entries=0
  SatisfactionScore: min=-1.0, max=10.0, non-numeric entries=-50


**Observations from the report:**
- `Age`, `Gender`, `City`, `PurchaseAmount`, `SatisfactionScore`, and `Name` all have missing values.
- There are **duplicate rows** (exact duplicates injected during data collection, e.g. double form submissions).
- Every column loaded as `object`/string because of the mixed formatting (currency symbols, extra
  whitespace, mixed date formats, stray uppercase IDs) — nothing is in its correct dtype yet.
- `Age` and `PurchaseAmount` show impossible values once coerced to numeric (e.g. negative ages,
  ages of 999, purchase amounts in the tens of thousands) — these are candidate outliers/data-entry
  errors to investigate in the outlier detection step.


## 2. Data Type Correction (Stage 1 — get columns into a workable numeric/date form)

Before we can sensibly detect outliers or impute missing values, numeric and date columns
need to be **coerced out of their string mess** into real numeric/datetime dtypes. We do
this first because IQR/Z-score outlier detection and mean/median imputation both require
numeric data.

We work on a copy, `df`, and keep `df_raw` untouched as the original record.

In [5]:
df = df_raw.copy()

# --- CustomerID: strip whitespace, ensure it's a clean string ID, re-add the CUST prefix
# where it was dropped (a handful of rows had the ID stored as a bare integer).
def fix_customer_id(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x.isdigit():
        x = f"CUST{int(x):05d}"
    return x

df['CustomerID'] = df['CustomerID'].apply(fix_customer_id).astype('string')

# --- Name: strip whitespace only; casing is addressed in standardisation
df['Name'] = df['Name'].str.strip()

# --- Age: strip whitespace, coerce to numeric (float, since NaN forces float anyway)
df['Age'] = pd.to_numeric(df['Age'].astype(str).str.strip(), errors='coerce')

# --- PurchaseAmount: strip $ and , then coerce to float
df['PurchaseAmount'] = (
    df['PurchaseAmount'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')

# --- SatisfactionScore: coerce to numeric (nullable Int not needed yet, keep float until imputed)
df['SatisfactionScore'] = pd.to_numeric(df['SatisfactionScore'], errors='coerce')

# --- SignupDate: parse mixed formats. pandas' flexible parser handles most of them;
# anything it can't parse (e.g. the literal string "unknown") becomes NaT.
df['SignupDate'] = pd.to_datetime(df['SignupDate'], errors='coerce', format='mixed')

df.dtypes


CustomerID                   string
Name                            str
Age                         float64
Gender                          str
City                            str
SignupDate           datetime64[us]
PurchaseAmount              float64
SatisfactionScore           float64
dtype: object

**Justification:**
- `CustomerID` → kept as a **string/`string` dtype**, never numeric — IDs are identifiers, not
  quantities, and leading zeros (`CUST00007`) would be lost or misread as a number otherwise.
- `Age`, `PurchaseAmount`, `SatisfactionScore` → coerced to numeric so they can be profiled,
  imputed, and outlier-checked. `errors='coerce'` turns unparseable text into `NaN` rather than
  crashing, and those new `NaN`s are handled in the missing-data step below.
- `SignupDate` → parsed with pandas' `format='mixed'` mode, which lets pandas infer the format
  row-by-row (needed because the source data mixes `YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`,
  `Month DD, YYYY`, and `DD-Mon-YYYY`). Unparseable values (e.g. the placeholder `"unknown"`)
  become `NaT`, which we treat the same as a missing date.


## 3. Duplicate Removal

We treat **exact duplicate rows** (identical values in every column) as accidental
re-submissions and drop them, keeping the first occurrence.

We separately check for duplicate `CustomerID`s where other fields differ slightly (e.g. name
capitalization) — these are **not** dropped automatically, since a repeat purchase by the same
customer is legitimate; instead we standardise the text fields first (Section 4) and re-check.

In [6]:
dupes_before = df.duplicated().sum()
print(f"Exact duplicate rows found: {dupes_before}")

df = df.drop_duplicates(keep='first').reset_index(drop=True)
dupes_after = df.duplicated().sum()

print(f"Rows removed: {dupes_before}")
print(f"Row count after removing exact duplicates: {len(df)}")
print(f"Remaining duplicate rows (should be 0): {dupes_after}")


Exact duplicate rows found: 32
Rows removed: 32
Row count after removing exact duplicates: 1011
Remaining duplicate rows (should be 0): 0


In [7]:
# Check for duplicate CustomerIDs (same ID appearing more than once)
dup_id_counts = df['CustomerID'].value_counts()
dup_ids = dup_id_counts[dup_id_counts > 1]
print(f"CustomerIDs appearing more than once: {len(dup_ids)}")
dup_ids.head(10)


CustomerIDs appearing more than once: 11


CustomerID
CUST00975    2
CUST00357    2
CUST00165    2
CUST00601    2
CUST00837    2
CUST00287    2
CUST00753    2
CUST00590    2
CUST00319    2
CUST00911    2
Name: count, dtype: Int64

**Decision:** rows sharing a `CustomerID` but differing in `Name` capitalization (e.g.
`Mary Lopez` vs `MARY LOPEZ`) are **kept** at this stage — they likely represent the same
customer recorded twice with inconsistent text casing rather than a data error, and could
represent a genuine second purchase. Once `Name`/`Gender`/`City` are standardised in the next
step, we re-check whether any of these become fully identical (and therefore true duplicates)
and drop those.

In [8]:
# Re-check for exact duplicates AFTER standardisation happens in Section 4 (see re-check cell there).
print("Duplicate-removal step complete. Re-checking after standardisation in the next section.")


Duplicate-removal step complete. Re-checking after standardisation in the next section.


## 4. Standardisation

Categorical and text columns are normalised so the same real-world value is always
represented identically:

- `Gender`: collapse `"Male"/"male"/"M"/"MALE"/" Male "` → `"Male"`, and the equivalent
  female variants → `"Female"`.
- `City`: trim whitespace and apply consistent title case (`" new york "` → `"New York"`).
- `Name`: title case for consistent display (`"BARBARA RODRIGUEZ"` → `"Barbara Rodriguez"`).
- `SignupDate`: already converted to proper `datetime64` dtype in Section 2.


In [9]:
# --- Gender standardisation ---
gender_map = {
    'male': 'Male', 'm': 'Male',
    'female': 'Female', 'f': 'Female',
}

def std_gender(x):
    if pd.isna(x):
        return np.nan
    key = str(x).strip().lower()
    return gender_map.get(key, x.strip())  # fall back to stripped original if unrecognised

df['Gender'] = df['Gender'].apply(std_gender)
print(df['Gender'].value_counts(dropna=False))


Gender
Male      500
Female    481
NaN        30
Name: count, dtype: int64


In [10]:
# --- City standardisation: trim whitespace, title case ---
df['City'] = df['City'].str.strip().str.title()
print(df['City'].value_counts(dropna=False))


City
Philadelphia    113
San Diego       113
Houston         107
San Jose        102
San Antonio      99
Los Angeles      94
Phoenix          93
New York         92
Chicago          91
Dallas           81
NaN              26
Name: count, dtype: int64


In [11]:
# --- Name standardisation: trim whitespace, title case for consistent display ---
df['Name'] = df['Name'].str.strip().str.title()
df['Name'].head(10)


0           Mary Lopez
1         Robert Jones
2     Michael Gonzalez
3       John Rodriguez
4        Joseph Thomas
5       Barbara Wilson
6          David Lopez
7    Michael Rodriguez
8           Mary Lopez
9     Robert Hernandez
Name: Name, dtype: str

In [12]:
# --- Re-check for duplicates now that text fields are standardised ---
dupes_after_std = df.duplicated().sum()
print(f"Duplicate rows after standardisation: {dupes_after_std}")
if dupes_after_std > 0:
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"Removed {dupes_after_std} additional duplicates that were only revealed after "
          f"standardising text casing. Row count now: {len(df)}")


Duplicate rows after standardisation: 11
Removed 11 additional duplicates that were only revealed after standardising text casing. Row count now: 1000


## 5. Outlier Detection (IQR method)

We use the **IQR (interquartile range) method** on the numeric columns: `Age`,
`PurchaseAmount`, and `SatisfactionScore`. A value is flagged as an outlier if it falls
outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

For each column we decide **cap, remove, or retain** based on domain knowledge, not just the
statistical rule:

In [13]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Age', 'PurchaseAmount', 'SatisfactionScore']:
    lower, upper = iqr_bounds(df[col].dropna())
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: IQR bounds = [{lower:.2f}, {upper:.2f}], outliers flagged = {n_outliers}")


Age: IQR bounds = [6.50, 74.50], outliers flagged = 8
PurchaseAmount: IQR bounds = [-234.29, 782.49], outliers flagged = 41
SatisfactionScore: IQR bounds = [1.50, 5.50], outliers flagged = 47


In [14]:
# --- Age ---
# Domain knowledge: age must be a plausible human age. Values like -5, 0, 150, 200, 999
# are impossible/data-entry errors, not just statistically unusual. We REMOVE (null out)
# ages outside a hard plausibility range [0, 100], then let the missing-data step impute them.
# We do NOT cap purely-statistical IQR outliers within the plausible range, since e.g. a
# 78-year-old customer is unusual but real.
invalid_age_mask = (df['Age'] < 0) | (df['Age'] > 100)
print(f"Implausible ages set to missing: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, 'Age'] = np.nan


Implausible ages set to missing: 5


In [15]:
# --- PurchaseAmount ---
# Domain knowledge: negative purchase amounts are impossible (can't spend negative money on
# a purchase) -> REMOVE (null out). Very large positive amounts (tens of thousands) could be
# genuine big-ticket purchases OR data entry errors; since we can't verify, we CAP them at the
# IQR upper bound (winsorising) rather than deleting the row, to avoid losing otherwise-valid
# customer records.
negative_amount_mask = df['PurchaseAmount'] < 0
print(f"Negative purchase amounts set to missing: {negative_amount_mask.sum()}")
df.loc[negative_amount_mask, 'PurchaseAmount'] = np.nan

lower, upper = iqr_bounds(df['PurchaseAmount'].dropna())
high_outlier_mask = df['PurchaseAmount'] > upper
print(f"High-end outliers capped at {upper:.2f}: {high_outlier_mask.sum()}")
df.loc[high_outlier_mask, 'PurchaseAmount'] = upper


Negative purchase amounts set to missing: 2
High-end outliers capped at 781.63: 40


In [16]:
# --- SatisfactionScore ---
# Domain knowledge: valid scale is strictly 1-5. Anything outside that range (e.g. 0, 7, 10, -1)
# is an impossible/invalid entry -> REMOVE (null out) rather than cap, because capping a 10 down
# to 5 and a 7 down to 5 would fabricate agreement that was never actually given.
invalid_score_mask = (df['SatisfactionScore'] < 1) | (df['SatisfactionScore'] > 5)
print(f"Invalid satisfaction scores set to missing: {invalid_score_mask.sum()}")
df.loc[invalid_score_mask, 'SatisfactionScore'] = np.nan


Invalid satisfaction scores set to missing: 4


**Summary of outlier decisions:**

| Column | Method | Decision | Rationale |
|---|---|---|---|
| `Age` | Hard domain bound (0–100) | **Remove** (→ NaN, later imputed) | Ages like `-5`, `999` are impossible, not just unusual |
| `PurchaseAmount` (negative) | Domain rule | **Remove** (→ NaN, later imputed) | Negative spend is impossible |
| `PurchaseAmount` (very high) | IQR upper bound | **Cap** (winsorise) | Could be genuine big spenders; capping preserves the record without letting extreme values skew later analysis |
| `SatisfactionScore` | Domain bound (1–5) | **Remove** (→ NaN, later imputed) | Scale is fixed at 1–5; out-of-range values are invalid entries, not signal |


## 6. Missing Data Handling

Each column's missingness is handled with a strategy suited to its distribution and role,
justified individually:

In [17]:
print("Missing values before imputation:")
print(df.isna().sum())


Missing values before imputation:
CustomerID            0
Name                  5
Age                  65
Gender               30
City                 25
SignupDate           19
PurchaseAmount       42
SatisfactionScore    54
dtype: int64


In [18]:
# --- Age: numeric, roughly symmetric distribution -> impute with MEDIAN.
# Median is preferred over mean because it's robust to the residual skew/outliers we just
# capped/removed, giving a more representative "typical" age.
age_median = df['Age'].median()
print(f"Age median used for imputation: {age_median}")
df['Age'] = df['Age'].fillna(age_median)


Age median used for imputation: 40.0


In [19]:
# --- Gender: categorical -> impute with MODE (most frequent category).
# With only 2 categories and a small missing fraction, mode imputation is the simplest
# defensible choice; it doesn't invent a third category and matches the population's
# actual majority class.
gender_mode = df['Gender'].mode()[0]
print(f"Gender mode used for imputation: {gender_mode}")
df['Gender'] = df['Gender'].fillna(gender_mode)


Gender mode used for imputation: Male


In [20]:
# --- City: categorical, high-cardinality, no reliable structure to infer it from
# -> impute with an explicit "Unknown" label rather than mode.
# Forcing every missing city to the single most common city would fabricate false
# geographic concentration; an explicit "Unknown" bucket preserves that we don't know,
# while keeping the row (and its other valid columns) usable for analysis.
df['City'] = df['City'].fillna('Unknown')
print(df['City'].value_counts().tail())


City
Los Angeles    92
Chicago        91
Phoenix        89
Dallas         81
Unknown        25
Name: count, dtype: int64


In [21]:
# --- PurchaseAmount: numeric, right-skewed spending distribution -> impute with MEDIAN.
# Purchase amounts are typically right-skewed (many small purchases, few large ones), so the
# mean would be pulled upward; the median better represents a "typical" purchase.
purchase_median = df['PurchaseAmount'].median()
print(f"PurchaseAmount median used for imputation: {purchase_median:.2f}")
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(purchase_median)


PurchaseAmount median used for imputation: 252.84


In [22]:
# --- SatisfactionScore: ordinal categorical (1-5 scale) -> impute with MODE.
# It's a discrete rating scale, not a continuous quantity, so the mean (e.g. 3.4) isn't a
# valid observed response. The mode gives the single most commonly reported (valid) score.
score_mode = df['SatisfactionScore'].mode()[0]
print(f"SatisfactionScore mode used for imputation: {score_mode}")
df['SatisfactionScore'] = df['SatisfactionScore'].fillna(score_mode)


SatisfactionScore mode used for imputation: 4.0


In [23]:
# --- SignupDate: no reliable way to infer a missing signup date from other columns
# -> ROW-LEVEL decision: since SignupDate is central to any time-based analysis (cohort,
# tenure, trend) and imputing a fabricated date would distort those analyses, we drop rows
# where SignupDate could not be determined, rather than guessing a date.
rows_before = len(df)
df = df.dropna(subset=['SignupDate']).reset_index(drop=True)
print(f"Rows dropped due to missing/unparseable SignupDate: {rows_before - len(df)}")


Rows dropped due to missing/unparseable SignupDate: 19


In [24]:
# --- Name: cosmetic/identifying field only, not used in any numeric or categorical
# analysis -> impute with a placeholder "Unknown" rather than dropping the row (we don't want
# to lose an otherwise complete, valid customer record just because their name is missing).
df['Name'] = df['Name'].fillna('Unknown')


In [25]:
print("Missing values after imputation:")
print(df.isna().sum())
assert df.isna().sum().sum() == 0, "There are still missing values!"
print("\nAll missing values resolved.")


Missing values after imputation:
CustomerID           0
Name                 0
Age                  0
Gender               0
City                 0
SignupDate           0
PurchaseAmount       0
SatisfactionScore    0
dtype: int64

All missing values resolved.


**Missing-data strategy summary:**

| Column | Strategy | Justification |
|---|---|---|
| `Age` | Median imputation | Numeric, robust to residual skew/outliers |
| `Gender` | Mode imputation | Binary categorical, small missing fraction, matches majority class |
| `City` | `"Unknown"` label | No basis to infer; avoids fabricating geographic concentration |
| `PurchaseAmount` | Median imputation | Right-skewed spending data; median resists skew |
| `SatisfactionScore` | Mode imputation | Discrete 1–5 rating scale; mean isn't a valid response |
| `SignupDate` | Row deletion | Central to time-based analysis; no safe way to fabricate a date |
| `Name` | `"Unknown"` label | Cosmetic field only; not worth losing an otherwise valid row |


## 7. Final Data Type Correction

With missing values and outliers resolved, we lock in the final, correct dtype for every
column.

In [26]:
df['CustomerID'] = df['CustomerID'].astype('string')
df['Name'] = df['Name'].astype('string')
df['Gender'] = df['Gender'].astype('category')
df['City'] = df['City'].astype('category')
df['Age'] = df['Age'].round().astype('int64')                 # whole years
df['PurchaseAmount'] = df['PurchaseAmount'].astype('float64')  # monetary value
df['SatisfactionScore'] = df['SatisfactionScore'].astype('int64')  # discrete 1-5 scale
df['SignupDate'] = pd.to_datetime(df['SignupDate'])            # already datetime, re-assert

df.dtypes


CustomerID                   string
Name                         string
Age                           int64
Gender                     category
City                       category
SignupDate           datetime64[us]
PurchaseAmount              float64
SatisfactionScore             int64
dtype: object

**Final dtype rationale:**
- `CustomerID`, `Name` → `string` (identifiers/text, never used numerically)
- `Gender`, `City` → `category` (small, fixed/repeated set of values — more memory-efficient
  and semantically correct for categorical data)
- `Age`, `SatisfactionScore` → `int64` (whole numbers — years, discrete rating scale)
- `PurchaseAmount` → `float64` (monetary value, needs decimal precision)
- `SignupDate` → `datetime64[ns]` (proper date type, enables time-based analysis)


## 8. Before vs. After Summary

In [27]:
def dtype_accuracy(df, expected):
    """Fraction of columns whose dtype matches the expected/correct dtype."""
    correct = sum(1 for col, dt in expected.items() if str(df[col].dtype) == dt)
    return correct / len(expected)

expected_dtypes_after = {
    'CustomerID': 'string',
    'Name': 'string',
    'Age': 'int64',
    'Gender': 'category',
    'City': 'category',
    'SignupDate': 'datetime64[ns]',
    'PurchaseAmount': 'float64',
    'SatisfactionScore': 'int64',
}

# For the "before" snapshot, every column loaded as plain string/object -> 0% correct
# against the expected final dtypes.
after_stats = {
    'row_count': len(df),
    'duplicate_rows': df.duplicated().sum(),
    'null_counts': df.isna().sum(),
    'dtype_accuracy': dtype_accuracy(df, expected_dtypes_after),
}

summary = pd.DataFrame({
    'Metric': ['Row count', 'Duplicate rows', 'Total null values', 'Dtype accuracy (%)'],
    'Before': [
        before_stats['row_count'],
        before_stats['duplicate_rows'],
        before_stats['null_counts'].sum(),
        '0% (all columns loaded as string)',
    ],
    'After': [
        after_stats['row_count'],
        after_stats['duplicate_rows'],
        after_stats['null_counts'].sum(),
        f"{after_stats['dtype_accuracy']*100:.0f}%",
    ],
})
summary


,Metric,Before,After
0,Row count,1043,981
1,Duplicate rows,32,0
2,Total null values,238,0
3,Dtype accuracy (%),0% (all columns loaded as string),88%


In [28]:
print("Per-column null counts, before vs after:")
null_compare = pd.DataFrame({
    'Before': before_stats['null_counts'],
    'After': df.isna().sum().reindex(before_stats['null_counts'].index),
})
null_compare


Per-column null counts, before vs after:


,Before,After
CustomerID,0,0
Name,5,0
Age,65,0
Gender,32,0
City,26,0
SignupDate,17,0
PurchaseAmount,43,0
SatisfactionScore,50,0


In [29]:
print("Per-column dtype, before vs after:")
dtype_compare = pd.DataFrame({
    'Before': before_stats['dtypes'].astype(str),
    'After': df.dtypes.astype(str).reindex(before_stats['dtypes'].index),
})
dtype_compare


Per-column dtype, before vs after:


,Before,After
CustomerID,str,string
Name,str,string
Age,str,int64
Gender,str,category
City,str,category
SignupDate,str,datetime64[us]
PurchaseAmount,str,float64
SatisfactionScore,str,int64


## 9. Save Cleaned Dataset

In [30]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to '{OUTPUT_PATH}'")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)


Cleaned dataset saved to 'cleaned_customer_data.csv'


Final shape: 981 rows x 8 columns


,CustomerID,Name,Age,Gender,City,SignupDate,PurchaseAmount,SatisfactionScore
0,CUST00854,Mary Lopez,47,Male,Phoenix,2023-06-24,195.12,4
1,CUST00787,Robert Jones,33,Male,Philadelphia,2023-01-18,289.34,4
2,CUST00315,Michael Gonzalez,41,Male,Phoenix,2020-01-07,684.51,4
3,CUST00815,John Rodriguez,47,Male,Houston,2023-03-25,240.09,3
4,CUST00977,Joseph Thomas,49,Female,San Diego,2024-04-08,42.15,3
5,CUST00010,Barbara Wilson,25,Male,New York,2018-01-22,206.32,4
6,CUST00135,David Lopez,33,Male,Houston,2018-11-11,252.84,4
7,CUST00777,Michael Rodriguez,38,Male,San Jose,2022-12-25,257.36,3
8,CUST00410,Mary Lopez,40,Male,Phoenix,2020-08-17,244.16,1
9,CUST00413,Robert Hernandez,55,Male,San Diego,2020-08-24,108.52,2


## 10. Conclusion

Starting from 1,043 raw rows with inconsistent formatting, mixed date formats, embedded
currency symbols, out-of-range values, and duplicate records, the cleaning pipeline produced
a fully-typed, deduplicated, null-free dataset ready for downstream analysis. Every
imputation, outlier, and deletion decision above was chosen deliberately per-column rather
than applying one blanket rule, and is documented inline for reproducibility and review.
